# C9-dimensionality-reduction — Session 2: Truncated SVD in Practice — Compressing the Embedding Stack

*One class session, roughly 85 minutes. Builds on Session 1 (the SVD
route, variance-explained thinking), F6-svd-spectral (thin SVD,
$\lVert W \rVert_F^2 = \sum_i \sigma_i^2$, the rank-$r$ error identity
$\lVert W - W_r \rVert_F^2 = \sum_{i>r} \sigma_i^2$, Eckart–Young), and
C8-embeddings (the unit-row stack `W`, the similarity matrix
$S = W W^{\mathsf T}$, the gensim cache register).*

**This session:** C8's embedding matrix, consumed **by name and by
convention** — `W`, shape $(N, 100)$, rows are tokens, rows
unit-normalized, $S = W W^{\mathsf T}$; the thin SVD of a real GloVe
stack; rank-$r$ truncation and its storage arithmetic; the **error
curve** — relative squared Frobenius error as a function of $r$ —
computed the cheap way (a cumulative sum over $\sigma^2$, no
re-decomposition); **choosing $r$ from an error budget**; and what
compression does to the similarity matrix, entry by entry and neighbor
by neighbor.

Everything runs on the fixed `glove-wiki-gigaword-100` artifact through
the shared cache, vectors cast to float64 at the load boundary — C8's
register unchanged.

Try every checkpoint by hand first, then verify in code.
Answers are collected at the end of this notebook.

In [ ]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import matplotlib.pyplot as plt

## 1. The Object Being Compressed: C8's `W`, by Name

C8 fixed a convention and promised it would be consumed as-is.
This is the consumption: the **unit-row embedding stack** `W` —
$N$ tokens as rows, shape $(N, 100)$, every row a unit vector, built by
the boundary-cast lookup and the broadcasting normalization — and its
Gram matrix $S = W W^{\mathsf T}$ of all pairwise cosines.

The working vocabulary: $48$ words in eight themed blocks of six, so
$S$ has structure worth compressing (blocks were what C8's heatmap made
visible; blocks are what a good low-rank approximation must keep).

In [ ]:
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

BLOCKS = {
    "weather":  ["storm", "breeze", "drizzle", "hail", "sunshine", "frost"],
    "kitchen":  ["oven", "skillet", "spoon", "kettle", "ladle", "whisk"],
    "sports":   ["soccer", "tennis", "hockey", "rugby", "golf", "cricket"],
    "body":     ["elbow", "knee", "shoulder", "ankle", "wrist", "thumb"],
    "vehicles": ["truck", "bus", "tram", "scooter", "van", "jeep"],
    "trees":    ["oak", "pine", "maple", "birch", "cedar", "willow"],
    "metals":   ["iron", "copper", "zinc", "nickel", "tin", "aluminum"],
    "emotions": ["joy", "anger", "fear", "sorrow", "delight", "envy"],
}
WORDS = [w for ws in BLOCKS.values() for w in ws]
N = len(WORDS)

V = np.asarray(kv[WORDS], dtype=np.float64)            # boundary cast, (48, 100)
norms = np.sqrt((V * V).sum(axis=1, keepdims=True))    # C8's register
W = V / norms
S = W @ W.T

print("W shape:", W.shape, "| dtype:", W.dtype)
assert np.allclose(np.sqrt((W * W).sum(axis=1)), 1.0, atol=1e-12, rtol=0)
assert np.allclose(S, S.T, atol=1e-12, rtol=0)
print("S shape:", S.shape, "| max |diag(S) - 1|:", np.max(np.abs(np.diag(S) - 1.0)))

Unit rows, symmetric $S$, unit diagonal — C8's three standing asserts,
passed.
(A register note: `np.linalg.norm` is a legal call in this unit and
would compute `norms` too; the broadcasting form stays in this course's
fingers because the exam still bans `np.linalg` in *some* problems, and
each problem states its own rules.)

Storage stakes, before any math: $S$ has $N^2$ entries but is built
from $W$'s $100N$ numbers — C8's point.
This session pushes further: can $rN$ numbers, $r \ll 100$, still carry
$S$?

### Checkpoint 1

1. Recite C8's convention from memory: what are `W`'s rows, its shape,
   its row norms, and the formula for $S$?
2. For $N = 48$: how many numbers are in $S$, and in $W$?
3. Why do the *themed blocks* matter for judging a compression — what
   should survive in $S_r$ if the compression is any good?

## 2. The Thin SVD of a Real Embedding Stack

F6's decomposition, on real data: $W = U \Sigma V^{\mathsf T}$ with
`full_matrices=False` — the default taught call, shapes
$(N, N), (N,), (N, 100)$ here since $N = 48 < 100$ (the wide case from
F6-03's shape table).
`np.linalg.svd` returns $\sigma$ **descending** — no reorder needed
(that `[::-1]` idiom belongs to `eigh`).

One identity gets a beautiful special case here.
F6-04 derived $\lVert W \rVert_F^2 = \sum_i \sigma_i^2$ for any matrix.
But *this* `W` has unit rows, so
$\lVert W \rVert_F^2 = \sum_{\text{rows}} \lVert w_i \rVert^2 = N$
exactly: **the singular values of a unit-row stack must square-sum to
$N = 48$.**

In [ ]:
U, s, Vt = np.linalg.svd(W, full_matrices=False)
print("shapes:", U.shape, s.shape, Vt.shape)
assert np.all(np.diff(s) <= 0)
print("sigma, first six:", np.round(s[:6], 4))
print("sigma, last three:", np.round(s[-3:], 4))

print("reconstruction gap:", np.max(np.abs(U @ np.diag(s) @ Vt - W)))
print("||W||_F^2:", (W * W).sum(), "   sum sigma^2:", np.round((s**2).sum(), 12))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(np.arange(1, N + 1), s, marker=".")
ax.set_xlabel("component index i")
ax.set_ylabel("singular value sigma_i")
ax.set_title("The spectrum of the 48-word stack: heavy head, long tail")
plt.tight_layout()
plt.show()

The spectrum tells the compression story before any truncation:
$\sigma_1 = 3.0344$ down to $\sigma_{48} = 0.1373$, a heavy head and a
long shallow tail — most of the square-sum $48$ lives in the first
couple dozen components.
Reconstruction from all $48$ components is exact to $10^{-15}$, and the
square-sum check lands on $48$ on the nose.

### Checkpoint 2

1. Give the thin-SVD shapes for a stack of $N = 300$ words (note
   $300 > 100$ — the tall case).
2. Without computing: a unit-row stack with $N = 20$ rows has
   $\sum_i \sigma_i^2 = {}$?
3. Which taught call would need the `[::-1]` reorder if you used it
   here by mistake, and why doesn't `np.linalg.svd`?

## 3. Rank-$r$ Truncation and What It Costs to Store

**Definition (F6-04, restated).**
Keep the top $r$ components:
$$W_r \;=\; U_{:, :r}\, \Sigma_r\, V^{\mathsf T}_{:r}
\qquad \text{(code: } \texttt{U[:, :r] @ np.diag(s[:r]) @ Vt[:r]}\text{)},$$
the best rank-$r$ approximation of $W$ in Frobenius norm
(Eckart–Young, stated in F6), with the **derived** error identity
$$\lVert W - W_r \rVert_F^2 \;=\; \sum_{i > r} \sigma_i^2 .$$
The **relative squared error** divides by
$\lVert W \rVert_F^2 = 48$ — a number between $1$ (keep nothing) and
$0$ (keep everything).

Storage: instead of $W$'s $N \cdot 100$ floats, the factors of $W_r$
need $r$ columns of $U$ ($rN$), $r$ singular values, and $r$ rows of
$V^{\mathsf T}$ ($100r$) — at $N = 48$, $r = 8$: $1192$ numbers versus
$4800$, a $4\times$ squeeze.
(The exam's setting is $N \gg 100$, where the $rN$ term dominates and
the squeeze is $100/r$.)

In [ ]:
def truncate(U, s, Vt, r):
    return U[:, :r] @ np.diag(s[:r]) @ Vt[:r]

fro2 = (W * W).sum()                                  # = 48.0
r = 8
Wr = truncate(U, s, Vt, r)
err_direct = np.sqrt(((W - Wr) ** 2).sum())           # measure the residual
err_tail = np.sqrt((s[r:] ** 2).sum())                # read it off the spectrum
print(f"r = {r}")
print(f"  error, direct route: {err_direct:.6f}")
print(f"  error, tail route  : {err_tail:.6f}")
print(f"  gap: {abs(err_direct - err_tail):.2e}")
print(f"  relative squared error: {err_direct**2 / fro2:.4f}")
print(f"  storage: {r * (N + 100 + 1)} numbers vs {N * 100} for full W")

The two routes agree to $10^{-16}$: measuring the residual matrix
entry-by-entry and summing eight-fewer squared singular values are the
same number, exactly as the identity promises.
That agreement is this unit's standing self-check — **every truncation
you compute should verify its error against the $\sigma$-tail** — and
it is how the practice problems ask you to prove your `W_r` is built
correctly.

At $r = 8$: relative squared error $0.3134$ — a $4\times$ storage
squeeze at the price of $31\%$ of the energy.
Whether that price is acceptable is not a math question; it is a
*budget* question, and Section 5 makes it precise.

### Checkpoint 3

1. Write the slicing for $W_{12}$ from `U, s, Vt` — which axis of each
   factor gets cut?
2. From the spectrum alone: what is
   $\lVert W - W_{47} \rVert_F$? (One singular value survives the
   tail: $\sigma_{48} = 0.1373$.)
3. For the exam's tall case ($N = 5000$, $d = 100$, $r = 10$): how
   many numbers do the truncated factors need, roughly, and which term
   dominates?

## 4. The Error Curve, the Cheap Way

The error identity turns "recompute the truncation for every $r$" into
one cumulative sum.
Since $\sum_{i>r}\sigma_i^2 = \lVert W\rVert_F^2 - \sum_{i \le r}\sigma_i^2$:

```python
rel_err2 = (fro2 - np.cumsum(s**2)) / fro2      # entry r-1 is rank-r's error
```

— all $48$ relative squared errors from one pass over the spectrum, no
truncation ever materialized.
Index carefully: `rel_err2[r - 1]` is rank $r$'s error (arrays start at
$0$, ranks start at $1$).

In [ ]:
rel_err2 = (fro2 - np.cumsum(s**2)) / fro2
for r_show in (1, 2, 4, 8, 16, 24, 32, 40, 48):
    print(f"  r = {r_show:2d}: relative squared error {rel_err2[r_show - 1]:.4f}")
assert np.isclose(rel_err2[7], 0.3134, atol=1e-4, rtol=0)      # Section 3's r=8

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(np.arange(1, N + 1), rel_err2, marker=".")
ax.set_xlabel("kept rank r")
ax.set_ylabel("relative squared Frobenius error")
ax.set_title("The error curve of the 48-word stack")
plt.tight_layout()
plt.show()

Read the curve like a price list: steep early drops mean the first
components are each buying a lot of accuracy ($0.8082$ at $r=1$ down to
$0.3134$ by $r=8$), and the long flat tail means the last twenty
components are each nearly worthless ($0.0087$ by $r=40$).
The curve ends at exactly $0$ ($r = 48$ keeps everything) and is
monotone decreasing — each kept component removes a $\sigma_i^2 \ge 0$
from the tail.

### Checkpoint 4

1. Why is the error curve monotone non-increasing in $r$ — which
   quantity leaves the tail at each step?
2. What are `rel_err2[0]` and `rel_err2[N - 1]`, in formula and (for
   this stack) in value?
3. A teammate's curve for the same stack starts at $0.19$ at $r = 1$
   and *increases* with $r$. Which indexing or cumsum mistake produces
   exactly that shape? (Hint: what does `np.cumsum(s**2)/fro2` itself
   look like?)

## 5. Choosing $r$ from an Error Budget

The deliverable in practice is almost never "the curve"; it is a
*decision*: **the smallest $r$ whose relative squared error is within a
stated budget $B$.**
On a descending error curve that is one idiom (C8's `argmax`-on-boolean
trick, resurfacing):

```python
r_star = int(np.argmax(rel_err2 <= B) + 1)      # first True, rank-indexed
```

`rel_err2 <= B` is a boolean array that flips from `False` to `True`
exactly once (monotone curve); `argmax` returns the first `True`; the
`+ 1` converts array index to rank.
A chosen $r^\*$ carries a two-sided certificate: **$r^\*$ meets the
budget and $r^\* - 1$ does not** — assert both, every time.

In [ ]:
for B in (0.30, 0.10, 0.05, 0.01):
    r_star = int(np.argmax(rel_err2 <= B) + 1)
    assert rel_err2[r_star - 1] <= B                       # meets the budget
    assert r_star == 1 or rel_err2[r_star - 2] > B         # minimally so
    print(f"  budget {B:.2f}: r* = {r_star:2d}   "
          f"(error {rel_err2[r_star - 1]:.4f}, at r*-1 it was {rel_err2[r_star - 2]:.4f})")

A $31\%$-error compression ($r=9$ for budget $0.30$) costs nine
components; pushing the budget to $1\%$ costs $40$ — the flat tail
makes tight budgets expensive.
This table is the honest summary of the whole trade: the curve says
what accuracy costs, the budget says what you are willing to pay,
$r^\*$ is the checkout.

### Checkpoint 5

1. Why does the first-`True` `argmax` idiom require the curve to be
   monotone? What could go wrong on a non-monotone array?
2. From Section 4's printed table, find $r^\*$ for budget $B = 0.16$
   by eye (between which two printed rows must it fall?).
3. Write the two-sided certificate asserts for a claimed
   $r^\* = 21$ at $B = 0.10$.

## 6. What Compression Does to $S$

The similarity matrix downstream of a truncation is
$S_r = W_r W_r^{\mathsf T}$ — the same formula, fed the compressed
stack.
How wrong is it?
Two probes, one global and one local:

- **entrywise**: $\max_{ij} |S - S_r|$ — the worst cosine error
  anywhere in the matrix;
- **behavioral**: do nearest-neighbor lists (C8's ranked retrieval)
  survive the compression?

One structural fact to anticipate: the rows of $W_r$ are **no longer
unit vectors** — truncation shortens them
($\lVert (W_r)_i \rVert \le 1$) — so $S_r$'s diagonal sags below $1$,
and its entries are only *approximate* cosines.

In [ ]:
for r_show in (8, 24):
    Wr_ = truncate(U, s, Vt, r_show)
    Sr_ = Wr_ @ Wr_.T
    gaps = np.abs(S - Sr_)
    print(f"r = {r_show:2d}: max entry gap {np.max(gaps):.4f}   "
          f"max diagonal gap {np.max(np.abs(np.diag(Sr_) - 1.0)):.4f}")

At both ranks the worst entry error *is* the worst diagonal error
($0.7103$ at $r = 8$, $0.1311$ at $r = 24$): the diagonal is where the
lost row-length shows up first, exactly as predicted — self-similarity
is the hardest thing to fake with few components.
Off-diagonal structure fares much better, as the neighbor probe
shows.

In [ ]:
r = 8
Wr = truncate(U, s, Vt, r)
Sr = Wr @ Wr.T

def top3(Smat, word):
    i = WORDS.index(word)
    order = np.argsort(Smat[i])[::-1]
    top = order[order != i][:3]                      # C8's self-exclusion idiom
    return [(WORDS[j], float(np.round(Smat[i, j], 4))) for j in top]

print("copper's neighbors, exact S :", top3(S, "copper"))
print("copper's neighbors, r=8 S_r :", top3(Sr, "copper"))

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.6))
for ax, M, title in [(axes[0], S, "S (exact)"), (axes[1], Sr, "S_r at r = 8")]:
    im = ax.imshow(M, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="(approximate) cosine", shrink=0.85)
plt.show()

With $r = 8$ of $48$ components — a third of the energy discarded —
`copper` still finds `zinc` first and `nickel` second, with values off
by under $0.01$; only third place wobbles (`aluminum` $0.7087$ swaps
with `iron` $0.7456$, a genuinely close call in the exact matrix too).
And the heatmaps tell the block story: the eight warm diagonal blocks
survive at $r = 8$; what fades is the fine off-block texture and the
hot diagonal (the sagging self-similarity from above).

This is the low-rank bet, stated once and carried into practice:
**coarse structure lives in the top components; the tail holds detail
(and noise).**

### Checkpoint 6

1. Why does truncation make rows *shorter*, never longer?
   (Think F6: $\lVert (W_r)_i \rVert^2$ sums squared coordinates over
   kept components only.)
2. Which entries of $S$ did $r = 8$ hurt most — diagonal or
   off-diagonal — and by how much at worst?
3. A ranking task needs only the *order* of each row of $S_r$, not its
   values. Judging from `copper`, is a small-$r$ compression more
   trustworthy for values or for order?

## 7. Worked Exam-Style Example: Constrained Coding

The exam register, worked in full: exact contract, exact bans, then the
solution and its self-checks.

---

**Problem.**
For the $48$-word stack `W` above (given), set

- `s48` — the singular values of `W` (thin SVD, descending as
  returned);
- `rel_curve` — the length-$48$ array of relative squared Frobenius
  errors, `rel_curve[r - 1]` for rank $r$, computed **from the
  spectrum, without building any $W_r$**;
- `r_budget` — the smallest rank whose relative squared error is at
  most $0.15$ (an `int`);
- `check` — `float(rel_curve[r_budget - 1])`.

**Allowed:** `np.linalg.svd`, `@`, `.T`, broadcasting, `np.cumsum`,
`np.argmax`.
**Banned (zero points): sklearn (any module), scipy, any Python loop
or comprehension over array elements, `np.einsum`, `np.tensordot`.**

---

**Step 1 — read the contract.**
"From the spectrum, without building any $W_r$" pins the route: the
error identity + cumsum, not $48$ truncations (a loop, and banned
anyway).

**Step 2 — the four lines.**

In [ ]:
_, s48, _ = np.linalg.svd(W, full_matrices=False)
rel_curve = ((s48**2).sum() - np.cumsum(s48**2)) / (s48**2).sum()
r_budget = int(np.argmax(rel_curve <= 0.15) + 1)
check = float(rel_curve[r_budget - 1])

print("r_budget:", r_budget, "| check:", f"{check:.4f}")
assert rel_curve[r_budget - 1] <= 0.15 and rel_curve[r_budget - 2] > 0.15

**Step 3 — certify.**
The two-sided assert passes: rank $17$ meets the $0.15$ budget
(error $0.1400$) and rank $16$ does not ($0.1529$, Section 4's table).
Note the graded habits: `int(...)` and `float(...)` casts to match the
stated types, the `+ 1` rank conversion, and a self-check *inside* the
submission.

### Checkpoint 7

1. Rework the decision at budget $0.20$ using Section 4's and 5's
   printed values: between which printed $r$ values must $r^\*$ lie,
   and does the Section 5 table settle it?
2. Which single banned tool would most naturally have crept into a
   "compute all 48 truncations" solution?

## 8. Common Pitfalls

**Pitfall 1 — reversing $\sigma$ but not the vectors (quiet,
catastrophic).**
The `eigh` habit — "outputs are ascending, reverse them" — applied
where it doesn't belong, or applied *halfway*: reversing `s` while
leaving `U, Vt` alone breaks the pairing between each $\sigma_i$ and
its directions.

In [ ]:
s_bad = s[::-1]                                   # "fixed" what wasn't broken
Wr_bad = U[:, :8] @ np.diag(s_bad[:8]) @ Vt[:8]
err_bad = np.sqrt(((W - Wr_bad) ** 2).sum())
print("error with mispaired sigma:", np.round(err_bad, 4))
print("error done right          :", np.round(np.sqrt(((W - truncate(U, s, Vt, 8)) ** 2).sum()), 4))
print("||W||_F itself            :", np.round(np.sqrt(fro2), 4))

No crash, plausible shapes — and an error of $6.46$ against
$\lVert W \rVert_F = 6.93$: the "approximation" is nearly as far from
$W$ as the zero matrix.
The tell: *always* verify the tail identity; the mispaired version
fails it by a mile.

**Pitfall 2 — `U[:8]` when you meant `U[:, :8]` (loud, luckily).**
Rows versus columns again: `U[:8]` takes eight *rows* (shape
$(8, 48)$), and the matmul chain refuses.

In [ ]:
try:
    U[:8] @ np.diag(s[:8]) @ Vt[:8]
except ValueError as e:
    print("ValueError:", e)

Read the two shapes in the message — $(8, 48) \times (8, 8)$ — and the
slip is obvious.
The loud failure is a *courtesy of this shape*: for a square stack
($N = 100$) the wrong slice would multiply through silently.
Never rely on the crash; rely on the tail-identity check.

**Pitfall 3 — treating $S_r$'s entries as exact cosines (quiet).**
Section 6 measured it: the diagonal sags ($0.7103$ worst gap at
$r = 8$), rows of $W_r$ are sub-unit, and every $S_r$ entry is an
approximation.
Any code that *asserts* `diag(S_r) == 1`, or normalizes by it
implicitly, imports a false invariant from the uncompressed world.

**Pitfall 4 — absolute vs relative budgets (quiet).**
"Error at most $0.05$" means nothing until you say *which* error:
relative squared ($0.05$ of $48$) or absolute squared ($0.05$ of
energy, full stop).
The same words, wildly different $r^\*$:

In [ ]:
abs_err2 = fro2 - np.cumsum(s**2)                 # absolute squared error
r_rel = int(np.argmax(rel_err2 <= 0.05) + 1)
r_abs = int(np.argmax(abs_err2 <= 0.05) + 1)
print("budget 0.05 read as relative:", r_rel)
print("budget 0.05 read as absolute:", r_abs)

$r^\* = 28$ versus $r^\* = 46$ — a factor-of-$1.6$ storage difference
from one ambiguous sentence.
Practice statements in this course always say "relative squared
Frobenius error"; when you write your own reports, be as pedantic.

### Checkpoint 8

1. Which pitfall does the standing tail-identity self-check catch, and
   what value pattern exposes it?
2. Why is pitfall 2 loud for this $48 \times 100$ stack but quiet for
   a square one?
3. In one sentence: why must every $S_r$ diagonal entry be $\le 1$?

## Exam Connections

How this session's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic tables — no real test text here):

- The exam's **dominant multi-part arc** runs embeddings → similarity
  → SVD → low-rank approximation: its back half is this session —
  `np.linalg.svd` on the unit-row stack (the arc's own hint names the
  call), truncated reconstruction, and a relative-squared-Frobenius
  error analysis over a range of ranks.
- **Proof sub-parts** ask for Frobenius-norm expressions in terms of
  singular values with "Reasoning is required" — Session 2's identity
  chain ($\lVert W\rVert_F^2 = \sum \sigma_i^2$, the tail identity,
  and p12's $\lVert S - S_r\rVert_F^2 = \sum_{i>r} \sigma_i^4$) is
  that register.
- **Grading signals**: per-problem API allowances (one part bans
  `np.linalg`, another recommends `np.linalg.svd`) — always read the
  problem's own rules; this course's statements mirror that scoping.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Rows are tokens (in `WORDS` order), shape $(N, 100)$, every row
   unit-norm, $S = W W^{\mathsf T}$.
2. $S$: $48^2 = 2304$; $W$: $4800$.
3. The warm diagonal blocks (within-theme similarity). A compression
   that erases blocks has destroyed the structure retrieval depends
   on, whatever its Frobenius score.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $U$: $(300, 100)$, $\sigma$: $(100,)$, $V^{\mathsf T}$:
   $(100, 100)$ — the tall case caps at $d = 100$.
2. $20$ — unit rows each contribute exactly $1$ to
   $\lVert W\rVert_F^2$.
3. `np.linalg.eigh` (ascending eigenvalues); `np.linalg.svd`'s
   documented contract returns singular values descending.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `U[:, :12] @ np.diag(s[:12]) @ Vt[:12]` — columns of $U$, entries
   of $\sigma$, rows of $V^{\mathsf T}$.
2. $\lVert W - W_{47}\rVert_F = \sigma_{48} = 0.1373$.
3. $r(N + d + 1) = 10 \cdot 5101 = 51{,}010$ — the $rN = 50{,}000$
   term dominates; versus $500{,}000$ for full $W$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Moving from $r$ to $r+1$ removes $\sigma_{r+1}^2 \ge 0$ from the
   tail sum, so the error cannot rise.
2. `rel_err2[0]` $= 1 - \sigma_1^2/48 = 0.8082$;
   `rel_err2[N-1]` $= 0$ exactly (empty tail).
3. They plotted the *kept* energy fraction
   `np.cumsum(s**2)/fro2` — an increasing curve starting at
   $\sigma_1^2/48 = 0.19$ — i.e. forgot to subtract from $1$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. `argmax` returns the *first* `True`; on a non-monotone array a
   later dip below the budget could be missed, or an early spurious
   dip chosen.
2. Between $r = 16$ ($0.1529$) and $r = 24$ ($0.0729$) from Section
   4's table — and Section 7 settles it exactly: $r^\* = 17$ at
   budget $0.15$, so for $0.16$, $r^\*$ is $16$ or $17$ (it is
   whichever first dips under $0.16$; $0.1529 \le 0.16$, so
   $r^\* = 16$).
3. `assert rel_err2[20] <= 0.10` and `assert rel_err2[19] > 0.10`.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Each row's squared norm is a sum of squared coordinates in the
   orthonormal $V$-directions; truncation drops the terms for
   $i > r$, so the sum can only shrink.
2. The diagonal, worst gap $0.7103$ — self-similarity needs the row's
   full length.
3. Order: `copper`'s top-2 survive with values off by $< 0.01$, and
   the only rank change is a near-tie. Values (especially near the
   diagonal) drift much sooner.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Section 4 brackets it between $r = 8$ ($0.3134$) and $r = 16$
   ($0.1529$); Section 5's budget-$0.30$ row ($r^\* = 9$, error
   $0.2877$) narrows to $9 \le r^\* \le 16$; the exact answer needs
   the curve (it is $13$, the first rank whose error, $0.1993$, dips
   under $0.20$).
2. A Python loop over $r$ — `for r in range(1, 49): ...` building
   each $W_r$ — is both banned and $48\times$ the work.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 (mispaired $\sigma$): the direct error wildly exceeds
   the $\sigma$-tail prediction — $6.46$ vs $3.88$ here.
2. `U[:8]` has shape $(8, 48)$, which cannot left-multiply the
   $(8, 8)$ diagonal; for a square $U$ the shapes happen to fit and
   the wrong number flows on silently.
3. $(S_r)_{ii} = \lVert (W_r)_i \rVert^2$, and truncated rows have
   norm $\le 1$.

</details>